# S11 - Masked Language Models & RAG
## Exercises

### Exercise 1 (Easy)
Use BERT for masked language modeling (fill in the blank).

In [ ]:
from transformers import pipeline

# Create a fill-mask pipeline
# Predict: "Paris is the [MASK] of France."


In [1]:
from transformers import pipeline

# Crear pipeline de masked language modeling
fill_mask = pipeline("fill-mask", model="bert-base-uncased")

# Predecir la palabra oculta
result = fill_mask("Paris is the [MASK] of France.")

print("Predicciones:")
for pred in result:
    print(f"  {pred['token_str']} (score: {pred['score']:.3f})")

c:\Users\Alejandro Varela\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Alejandro Varela\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alejandro Varela\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mo

Predicciones:
  capital (score: 0.997)
  heart (score: 0.001)
  center (score: 0.000)
  centre (score: 0.000)
  city (score: 0.000)


### Exercise 2 (Easy)
Generate sentence embeddings using a sentence-transformer model.

In [ ]:
from sentence_transformers import SentenceTransformer

sentences = [
    "The cat sat on the mat.",
    "A dog is playing in the park.",
    "The feline rested on the rug."
]

# Generate embeddings and compute cosine similarities


In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Cargar modelo de embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "The cat sat on the mat.",
    "A dog is playing in the park.",
    "The feline rested on the rug."
]

# Generar embeddings
embeddings = model.encode(sentences)

# Calcular similitud coseno
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Matriz de similitud:")
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_sim(embeddings[i], embeddings[j])
        print(f"  {sentences[i][:20]} vs {sentences[j][:20]} → {sim:.3f}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6887.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Matriz de similitud:
  The cat sat on the m vs A dog is playing in  → 0.085
  The cat sat on the m vs The feline rested on → 0.549
  A dog is playing in  vs The feline rested on → 0.094


### Exercise 3 (Medium)
Build a simple semantic search using FAISS.

In [ ]:
import faiss
import numpy as np

documents = [
    "Python is a programming language.",
    "Machine learning uses statistical methods.",
    "Deep learning is a subset of machine learning.",
    "Natural language processing handles text data.",
    "Computer vision processes images."
]

# Create FAISS index and search for "How do neural networks work?"


In [3]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Documentos de ejemplo
documents = [
    "Python is a programming language.",
    "Machine learning uses statistical methods.",
    "Deep learning is a subset of machine learning.",
    "Natural language processing handles text data.",
    "Computer vision processes images."
]

# Modelo de embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generar embeddings y normalizarlos (para usar producto escalar = coseno)
doc_embeddings = model.encode(documents)
faiss.normalize_L2(doc_embeddings)  # normalización L2

# Crear índice FAISS (inner product = coseno tras normalizar)
dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings)

# Consulta
query = "How do neural networks work?"
query_emb = model.encode([query])
faiss.normalize_L2(query_emb)

# Buscar los top-2 más similares
k = 2
scores, indices = index.search(query_emb, k)

print(f"Consulta: '{query}'\n")
for i, idx in enumerate(indices[0]):
    print(f"Top {i+1}: '{documents[idx]}' (score = {scores[0][i]:.3f})")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8754.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Consulta: 'How do neural networks work?'

Top 1: 'Deep learning is a subset of machine learning.' (score = 0.434)
Top 2: 'Machine learning uses statistical methods.' (score = 0.420)


### Exercise 4 (Medium)
Implement a RAG pipeline using TF-IDF retrieval and a local LLM via Ollama.

In the previous exercises you used HuggingFace models for both retrieval and generation. Now you will build a RAG pipeline that connects to a **local LLM running on Ollama** (as you learned in S10) for the generation step. For retrieval, you will use a simple TF-IDF approach instead of FAISS — this removes the dependency on sentence-transformers and keeps the focus on the local LLM integration.

The knowledge base consists of short paragraphs about NLP concepts from this course.

**Task:**
1. Create a knowledge base with at least 6 paragraphs about NLP topics (tokenization, embeddings, attention, transformers, sentiment analysis, named entity recognition)
2. Implement a `retrieve(query, documents, top_k=2)` function that uses `TfidfVectorizer` and `cosine_similarity` from sklearn to find the most relevant documents
3. Implement a `generate_answer(query, context)` function that sends the retrieved context and the question to Ollama's `/api/generate` endpoint and returns the model's answer
4. Test your RAG pipeline with at least 3 different questions about NLP topics
5. For each question, print: the retrieved documents (with similarity scores), the model's answer, and whether the answer correctly uses the retrieved context

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import requests
import numpy as np

OLLAMA_URL = "http://localhost:11434"

# --- Knowledge Base ---
documents = [
    # Add at least 6 short paragraphs about NLP concepts
    # Example: "Tokenization is the process of breaking text into smaller units called tokens..."
]

# --- Retrieval Function ---
def retrieve(query, documents, top_k=2):
    # Use TF-IDF + cosine similarity to find the top_k most relevant documents
    pass

# --- Generation Function ---
def generate_answer(query, context):
    # Send the query and retrieved context to Ollama's /api/generate endpoint
    # Return the model's answer
    pass

# --- Test the RAG Pipeline ---
questions = [
    "What is the attention mechanism in transformers?",
    "How does tokenization work in NLP?",
    "What is sentiment analysis used for?",
]

# For each question: retrieve context, generate answer, print results

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import requests
import numpy as np

OLLAMA_URL = "http://localhost:11434"

# --- Base de conocimiento (al menos 6 párrafos) ---
documents = [
    "Tokenization is the process of breaking text into smaller units called tokens, which can be words, subwords, or characters.",
    "Word embeddings are dense vector representations of words that capture semantic meaning, such as Word2Vec and GloVe.",
    "The attention mechanism allows a model to focus on relevant parts of the input sequence when making predictions, which is key to transformers.",
    "Transformers are neural network architectures that rely entirely on self-attention, avoiding recurrence and convolutions.",
    "Sentiment analysis is a task that classifies the emotional tone of a text (positive, negative, neutral) using NLP techniques.",
    "Named Entity Recognition (NER) identifies and classifies entities like persons, organizations, locations, and dates in text."
]

# --- Función de recuperación TF-IDF ---
def retrieve(query, documents, top_k=2):
    # Crear matriz TF-IDF para los documentos
    vectorizer = TfidfVectorizer()
    doc_tfidf = vectorizer.fit_transform(documents)
    
    # Transformar la consulta
    query_tfidf = vectorizer.transform([query])
    
    # Calcular similitud coseno
    similarities = cosine_similarity(query_tfidf, doc_tfidf).flatten()
    
    # Obtener índices de los top_k documentos
    top_indices = np.argsort(similarities)[::-1][:top_k]
    results = [(documents[i], similarities[i]) for i in top_indices]
    return results

# --- Función de generación con Ollama ---
def generate_answer(query, context):
    # Construir el prompt con el contexto recuperado
    context_str = "\n\n".join([doc for doc, _ in context])
    prompt = f"""Usa solo la siguiente información para responder la pregunta.
Si no encuentras la respuesta, di que no la sabes.

Información:
{context_str}

Pregunta: {query}

Respuesta:"""
    
    payload = {
        "model": "gemma3:1b",   # o "tinyllama"
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.2}
    }
    response = requests.post(f"{OLLAMA_URL}/api/generate", json=payload)
    if response.status_code == 200:
        return response.json()["response"].strip()
    else:
        return f"Error: {response.status_code}"

# --- Probar el pipeline con 3 preguntas ---
questions = [
    "What is the attention mechanism in transformers?",
    "How does tokenization work in NLP?",
    "What is sentiment analysis used for?"
]

print("=== RAG Pipeline ===")
for q in questions:
    print(f"\n\nPregunta: {q}")
    retrieved_docs = retrieve(q, documents, top_k=2)
    print("Documentos recuperados:")
    for doc, score in retrieved_docs:
        print(f"  - (score {score:.3f}) {doc[:80]}...")
    
    answer = generate_answer(q, retrieved_docs)
    print(f"\nRespuesta generada:\n{answer}")
    
    # Verificar si la respuesta usa correctamente el contexto (simple comprobación)
    context_used = any(any(word in answer.lower() for word in doc.lower().split()[:5]) for doc, _ in retrieved_docs)
    print(f"¿Usa el contexto? {'Sí' if context_used else 'No (posible alucinación)'}")

=== RAG Pipeline ===


Pregunta: What is the attention mechanism in transformers?
Documentos recuperados:
  - (score 0.394) The attention mechanism allows a model to focus on relevant parts of the input s...
  - (score 0.182) Transformers are neural network architectures that rely entirely on self-attenti...

Respuesta generada:
The attention mechanism allows a model to focus on relevant parts of the input sequence when making predictions, which is key to transformers.
¿Usa el contexto? Sí


Pregunta: How does tokenization work in NLP?
Documentos recuperados:
  - (score 0.155) Sentiment analysis is a task that classifies the emotional tone of a text (posit...
  - (score 0.149) Named Entity Recognition (NER) identifies and classifies entities like persons, ...

Respuesta generada:
No sé.
¿Usa el contexto? No (posible alucinación)


Pregunta: What is sentiment analysis used for?
Documentos recuperados:
  - (score 0.422) Sentiment analysis is a task that classifies the emotional tone of a

### Exercise 5 (Hard)
Build a RAG-powered Q&A system with answer quality evaluation using a local LLM.

In this exercise, you will extend the RAG pipeline from Exercise 4 into a more complete Q&A system. You will create a set of questions with known (ground truth) answers, run them through your RAG pipeline, and evaluate the quality of the generated answers — both automatically and manually.

This simulates a real-world scenario where you need to assess whether a RAG system is reliable enough for production use.

**Task:**
1. Expand the knowledge base from Exercise 4 to at least 10 documents covering a broader range of NLP topics (add documents about word embeddings, RNNs, LSTMs, BERT, text classification, language modeling, etc.)
2. Create an evaluation dataset: a list of at least 8 question-answer pairs where the answer can be found in the knowledge base. Include at least 2 questions whose answer is **not** in the knowledge base (to test how the system handles missing information)
3. Run each question through the RAG pipeline and collect the generated answers
4. Implement two evaluation methods:
   - **Automatic:** Use Ollama itself as a judge — send the question, ground truth answer, and generated answer to the model and ask it to rate the answer on a scale of 1-5 (faithfulness to context, correctness, completeness)
   - **Manual:** For each answer, print the question, retrieved context, generated answer, and ground truth side by side so you can inspect the results
5. Print a summary with the average score across all questions and identify which questions the system struggled with

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import requests
import numpy as np

OLLAMA_URL = "http://localhost:11434"

# --- Expanded Knowledge Base ---
documents = [
    # Add at least 10 NLP topic paragraphs
]

# --- Evaluation Dataset ---
eval_data = [
    # ("question", "ground_truth_answer"),
    # Include at least 2 questions with no answer in the knowledge base
]

# --- Reuse or redefine retrieve() and generate_answer() from Exercise 4 ---

# --- LLM-as-Judge Evaluation ---
def evaluate_answer(question, ground_truth, generated_answer):
    # Send to Ollama: ask it to rate the generated answer vs ground truth (1-5)
    # Return the score
    pass

# --- Run Evaluation ---
# For each question: retrieve, generate, evaluate, print results

# --- Print Summary ---
# Average score, worst-performing questions, observations

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import requests
import numpy as np

OLLAMA_URL = "http://localhost:11434"

# --- Base de conocimiento ampliada (≥10 documentos) ---
documents = [
    "Tokenization is the process of breaking text into smaller units called tokens, which can be words, subwords, or characters.",
    "Word embeddings are dense vector representations of words that capture semantic meaning, such as Word2Vec and GloVe.",
    "The attention mechanism allows a model to focus on relevant parts of the input sequence when making predictions, which is key to transformers.",
    "Transformers are neural network architectures that rely entirely on self-attention, avoiding recurrence and convolutions.",
    "Sentiment analysis is a task that classifies the emotional tone of a text (positive, negative, neutral) using NLP techniques.",
    "Named Entity Recognition (NER) identifies and classifies entities like persons, organizations, locations, and dates in text.",
    "RNNs (Recurrent Neural Networks) process sequences by maintaining a hidden state that captures information from previous time steps.",
    "LSTMs (Long Short-Term Memory) are a type of RNN designed to remember information for long periods, avoiding the vanishing gradient problem.",
    "BERT is a masked language model pre-trained on large text corpora using a bidirectional transformer architecture.",
    "Text classification assigns predefined categories to text documents, such as spam detection or topic labeling."
]

# --- Dataset de evaluación (8+ pares, incluyendo 2 sin respuesta) ---
eval_data = [
    ("What is tokenization?", "Tokenization is the process of breaking text into smaller units called tokens."),
    ("What does the attention mechanism do?", "The attention mechanism allows a model to focus on relevant parts of the input sequence."),
    ("What is BERT?", "BERT is a masked language model pre-trained using a bidirectional transformer architecture."),
    ("What are LSTMs used for?", "LSTMs are used to remember information for long periods, avoiding the vanishing gradient problem."),
    ("What is sentiment analysis?", "Sentiment analysis classifies the emotional tone of a text as positive, negative, or neutral."),
    ("Name one application of text classification.", "Text classification can be used for spam detection or topic labeling."),
    # Preguntas sin respuesta en la base de conocimiento
    ("What is the capital of France?", "The knowledge base does not contain this information."),
    ("Who wrote the novel 'Don Quixote'?", "The knowledge base does not contain this information.")
]

# --- Reutilizar retrieve y generate_answer (con pequeña modificación) ---
def retrieve(query, documents, top_k=2):
    vectorizer = TfidfVectorizer()
    doc_tfidf = vectorizer.fit_transform(documents)
    query_tfidf = vectorizer.transform([query])
    similarities = cosine_similarity(query_tfidf, doc_tfidf).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(documents[i], similarities[i]) for i in top_indices]

def generate_answer(query, context):
    context_str = "\n\n".join([doc for doc, _ in context])
    prompt = f"""Usa solo la siguiente información para responder la pregunta.
Si no encuentras la respuesta, responde exactamente: "No sé".
No inventes información.

Información:
{context_str}

Pregunta: {query}

Respuesta:"""
    payload = {
        "model": "gemma3:1b",
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.2}
    }
    response = requests.post(f"{OLLAMA_URL}/api/generate", json=payload)
    if response.status_code == 200:
        return response.json()["response"].strip()
    return "Error"

# --- Evaluación con LLM como juez ---
def evaluate_answer(question, ground_truth, generated_answer):
    prompt = f"""Evalúa la calidad de la respuesta generada para la pregunta dada.
Compara con la respuesta correcta (ground truth).
Asigna una puntuación del 1 al 5 según:
5 = Respuesta perfecta, coincide completamente con la verdad.
4 = Muy buena, pequeñas omisiones.
3 = Aceptable, pero falta información importante.
2 = Mala, contiene errores.
1 = Muy mala, irrelevante o inventada.

Pregunta: {question}
Respuesta correcta: {ground_truth}
Respuesta generada: {generated_answer}

Devuelve SOLO el número (1-5)."""

    payload = {
        "model": "gemma3:1b",
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0}
    }
    response = requests.post(f"{OLLAMA_URL}/api/generate", json=payload)
    if response.status_code == 200:
        try:
            score = int(response.json()["response"].strip())
            return max(1, min(5, score))
        except:
            return 3  # valor por defecto
    return 3

# --- Ejecutar evaluación completa ---
print("=== Evaluación del sistema RAG ===\n")
results = []
for question, gt in eval_data:
    print(f"Pregunta: {question}")
    retrieved = retrieve(question, documents, top_k=2)
    print(f"  Documentos recuperados (similitud):")
    for doc, sim in retrieved:
        print(f"    - ({sim:.3f}) {doc[:70]}...")
    
    answer = generate_answer(question, retrieved)
    print(f"  Respuesta generada: {answer[:200]}")
    print(f"  Ground truth: {gt[:200]}")
    
    score = evaluate_answer(question, gt, answer)
    results.append((question, score, answer, gt))
    print(f"  Puntuación (LLM-as-judge): {score}/5\n")
    print("-" * 70)

# --- Resumen final ---
scores = [s for _, s, _, _ in results]
avg_score = np.mean(scores)
print("\n=== RESUMEN ===")
print(f"Puntuación promedio: {avg_score:.2f}/5")
print("\nPreguntas con peor rendimiento (puntuación < 3):")
for q, s, a, gt in results:
    if s < 3:
        print(f"  - '{q}' (score {s})")
        print(f"      Esperado: {gt[:100]}")
        print(f"      Generado: {a[:100]}")

=== Evaluación del sistema RAG ===

Pregunta: What is tokenization?
  Documentos recuperados (similitud):
    - (0.305) Tokenization is the process of breaking text into smaller units called...
    - (0.102) BERT is a masked language model pre-trained on large text corpora usin...
  Respuesta generada: Tokenization is the process of breaking text into smaller units called tokens, which can be words, subwords, or characters.
  Ground truth: Tokenization is the process of breaking text into smaller units called tokens.
  Puntuación (LLM-as-judge): 5/5

----------------------------------------------------------------------
Pregunta: What does the attention mechanism do?
  Documentos recuperados (similitud):
    - (0.403) The attention mechanism allows a model to focus on relevant parts of t...
    - (0.142) Transformers are neural network architectures that rely entirely on se...
  Respuesta generada: Allows a model to focus on relevant parts of the input sequence when making predictions,